# Thomson-1.0-Small

Seven-arm experiment on [`thomsonreuters/Thomson-1.0-Small`](https://huggingface.co/thomsonreuters/Thomson-1.0-Small). Isolated from the main sweep on size: 70.2 GB weights, A100 80GB (High RAM).

Checkpoint card: base `tri-fair-lab/Snowdon1.1-Small`, architecture `Qwen3_5MoeForConditionalGeneration`. The model is a shipped legal-product weight, post-trained (Constitutional DPO / conformance RL); that may move deference relative to the open bases.


## Licence

Thomson-1.0-Small is **PolyForm Strict 1.0.0**. Permitted: research, experiment, and testing for public knowledge; use by educational and public research organisations; fair use. No restriction on benchmarking or publishing results. Forbidden: distributing the software or making derivative works — this notebook does neither.

The grant is for a noncommercial purpose. An open paper qualifies; use in service of a commercial offering is a separate question.

Full text: https://polyformproject.org/licenses/strict/1.0.0


## 1. Clone


In [ ]:
import os, sys, json, pathlib, subprocess

REPO_URL = "https://github.com/ryanmcdonough/behaviour-microscope.git"
REPO_DIR = pathlib.Path("/content/behaviour-microscope")

if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    dirty = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--porcelain"],
                           capture_output=True, text=True).stdout.strip()
    if dirty:
        print("Local changes — stashing:\n" + dirty)
        !git -C $REPO_DIR stash -u
    !git -C $REPO_DIR fetch --depth 1 origin main -q
    !git -C $REPO_DIR reset --hard origin/main -q

os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
!git -C $REPO_DIR log --oneline -1

## 2. Install


In [ ]:
%pip install -q -e '/content/behaviour-microscope'

# Colab has shipped torch and torchaudio built against different CUDA versions (13.0 against
# 12.8 on 2026-09-04). transformers imports torchaudio lazily while resolving a model class and
# rewrites the failure as a missing class, so the skew surfaces four cells down as
# `ModuleNotFoundError: Could not import module 'Qwen3_5MoeForCausalLM'` -- the class is
# present, its import chain is not. Nothing here decodes audio, so an unimportable torchaudio is
# dropped rather than rebuilt. A no-op on a runtime whose pair is already consistent.
import subprocess, sys

_probe = subprocess.run([sys.executable, "-c", "import torchaudio"], capture_output=True, text=True)
if _probe.returncode:
    print("torchaudio does not import:", (_probe.stderr.strip().splitlines() or ["?"])[-1])
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
    sys.modules.pop("torchaudio", None)
    print("removed torchaudio -- unused here, and transformers skips it once it is gone")
    if "transformers" in sys.modules:
        print("\nRESTART REQUIRED: transformers is already imported and has cached torchaudio as\n"
              "available. Runtime -> Restart session, then run from cell 1.")
else:
    print("torchaudio imports cleanly")

print("installed")

## 3. Hardware

Weights are 70.2 GB. The 40GB A100 will OOM during load; this cell exits first.


In [ ]:
import torch

WEIGHTS_GB = 70.2
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → GPU."
props = torch.cuda.get_device_properties(0)
total = props.total_memory / 1e9
print(f"{props.name}  |  {total:.1f} GB  |  compute {props.major}.{props.minor}")

headroom = total - WEIGHTS_GB
if headroom < 4:
    raise SystemExit(
        f"This card has {total:.0f} GB; weights are {WEIGHTS_GB} GB. "
        "Use the A100-80GB (High RAM) runtime.")
print(f"Headroom after weights: ~{headroom:.0f} GB")

DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
DEVICE = "cuda"
free_disk = os.statvfs('/content').f_bavail * os.statvfs('/content').f_frsize / 1e9
print(f"dtype: {DTYPE}   device: {DEVICE}   free disk: {free_disk:.0f} GB (need ~{WEIGHTS_GB:.0f})")


## 4. Compatibility

Config and class import only — **do not instantiate**. `from_config()` on this 35B MoE, even
under `torch.device("meta")`, builds every expert module and fills Colab host RAM. The real
load in 6b then OOMs. Point resolution happens at load; this cell only checks that the
architecture is importable (the torchaudio failure mode) before a 70 GB download.


In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "thomsonreuters/Thomson-1.0-Small"

cfg = AutoConfig.from_pretrained(MODEL_ID)
arch = (cfg.architectures or ["?"])[0]
print("architecture:", arch)

# Import the mapped class; do not call from_config / EagerModel. Instantiating this tree is
# what filled RAM in the previous version of this cell.
try:
    cls = AutoModelForCausalLM._model_mapping.get(type(cfg))
    if cls is None:
        import transformers as _T
        cls = getattr(_T, arch)
except Exception as exc:
    raise SystemExit(
        f"{type(exc).__name__}: {exc}\n\n"
        "transformers reports an import failure *inside* a modeling module as a missing class, so "
        "this is usually a broken optional dependency rather than an unsupported architecture -- "
        "the line above printed the architecture, so the config itself parsed. Re-run the install "
        "cell (its torchaudio guard), restart the runtime, and come back."
    ) from exc

text = getattr(cfg, "text_config", cfg)
print("causal-lm class:", cls.__name__)
print(f"layers: {getattr(text, 'num_hidden_layers', '?')}   "
      f"d_model: {getattr(text, 'hidden_size', '?')}")
n_exp = getattr(text, "num_experts", None) or getattr(text, "num_local_experts", None)
if n_exp is not None:
    print(f"experts: {n_exp}   active: {getattr(text, 'num_experts_per_tok', '?')}")

tok = AutoTokenizer.from_pretrained(MODEL_ID)
print("chat template:", "present" if tok.chat_template else "MISSING")

reserved = torch.cuda.memory_reserved() / 1e9
print(f"GPU reserved: {reserved:.2f} GB  (expect ~0; this cell must not load weights)")
if reserved > 1:
    print("WARNING: GPU memory is already occupied. Runtime → Restart session before 6b.")


## 5. Scenarios

Same 30 items, seven arms, and answer key as the other runs.


In [ ]:
from pathlib import Path
from microscope.experiment import (
    RunConfig, run_sweep, compare_runs, finalize_run, format_measurement_plan, resolve_run_dir,
)
from microscope.scenarios import ARMS, load_scenarios
import pandas as pd

scenarios = load_scenarios()
print(len(scenarios), "scenarios")
for arm in ARMS:
    print(f"  {arm.name:20s} {arm.cue or '(no assertion)'}")


## 6. Run

Thomson-1 reads `enable_thinking` and emits `<think>`:

1. Reasoning off — comparable to gemma / Qwen-off; mechanistic sweep available.
2. Reasoning on — behavioural only (answer is not at the final prompt position).

**Reasoning-off is switched off**, because `results/20260904T071909Z` already has it and that
run is unaffected by the parse fix: with reasoning off the answer is read from the first
token's logits, so the reasoning-block parser is never called. Set `RUN_PLAIN = True` to
reproduce it.

Reasoning-on is being re-run because `results/20260904T075505Z` is invalid: its completions
carried a closing `</think>` with no opening tag, so the block was never stripped and the
answer letter was read out of the echoed option list — 152 of 210 rows wrong, and a floor
accuracy of 43% that was an artifact. That run also used a 512-token budget and truncated 54%
of completions mid-thought without flagging them. Both fixed; the budget is now 2048.

Both arms run on the eager backend. vLLM would decode this faster, but installing it swaps the
torch it pins for Colab's, which is what left torchaudio built against the wrong CUDA and broke
the compatibility cell above; that is not worth trading a working runtime for. **Budget the
time**: reasoning-on took 4h50m at 512 tokens, and the cap is now four times that, so most of a
day is possible if completions use it. The preflight cell below spends one prompt confirming
the answer parses before committing to 210.

Each finished measurement is written to Drive before the next starts, so a Colab timeout
loses only the in-flight prompt. Resume with the same `RUN_NAME`; set `START_AT` to the
progress number you want next, or leave it `None` to continue after the last saved row.
Touch `STOP` in the run folder (Colab terminal) or interrupt the kernel to halt after the
current item, then run 6c to export plots from whatever is on disk — no model reload.


In [ ]:
MECHANISTIC = True     # False -> experiment 1 only, far faster on a 35B MoE
RUN_PLAIN = False      # reasoning-off: already in results/20260904T071909Z, unaffected by the fix
RUN_THINKING = True    # reasoning-on: re-run, the previous one mis-parsed every answer

# Eager for both arms. Capture and patching need the Python forward hooks that only this backend
# runs, and the vLLM alternative is off the table here -- see the note above. A reasoning run
# stays behavioural-only regardless: LocalBackend.supports_mechanistic_now gates on
# response_mode == "logits", so experiments 2-4 are skipped for it on any backend.
BACKEND = "eager"

# Persist off the ephemeral Colab disk. Timeouts wipe /content; they do not wipe Drive.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = Path("/content/drive/MyDrive/behaviour-microscope/results")
except Exception as exc:
    RESULTS_ROOT = Path("results")
    print(f"Drive not mounted ({type(exc).__name__}: {exc}); writing to {RESULTS_ROOT.resolve()}")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

START_AT = None        # 1-based, matches [N/210]. None = continue after rows already on disk.
FINALIZE_ONLY = False  # True = rebuild plots/tables from CSV, do not load the model.

def thomson(thinking):
    return RunConfig(
        model_id=MODEL_ID, provider="local", dtype=DTYPE, backend=BACKEND,
        extra_load_kwargs={"device": DEVICE},
        n_candidate_layers=4,
        enable_thinking=thinking,
        mechanistic=MECHANISTIC,   # a reasoning run is behavioural-only regardless
        results_root=RESULTS_ROOT,
        run_name="thomson-1-thinking" if thinking else "thomson-1-plain",
        start_at=START_AT,
        finalize_only=FINALIZE_ONLY,
        # arms=("floor", "junior_said", "partner_said", "partner_confirmed", "court"),
    )

configs = [thomson(thinking=t) for t, on in ((False, RUN_PLAIN), (True, RUN_THINKING)) if on]
if not configs:
    raise SystemExit("Nothing to run: set RUN_PLAIN or RUN_THINKING.")
for c in configs:
    path = resolve_run_dir(c)
    csv = path / "behavioural.csv"
    n = len(pd.read_csv(csv)) if csv.exists() else 0
    plan_n = len(scenarios) * len(c.arms)
    print(f"  reasoning {'on ' if c.enable_thinking else 'off'}  backend={c.backend}")
    print(f"    {n}/{plan_n} saved in {path}")
    if n < plan_n:
        from microscope.experiment import measurement_plan
        s, arm = measurement_plan(scenarios, c.arms)[n]
        print(f"    next is {n + 1}: {s.id} / {arm}")


### 6a. Preflight

One prompt through the configured backend, to confirm the answer parses before committing to
210 — the check run `20260904T075505Z` did not have, which is why its mis-parse was only
visible 4h50m later.

**Off by default.** It is a second 70 GB load. The previous session reported
`69.3 GB -> 69.3 GB` after teardown, and 6b then OOMs on load. Set `PREFLIGHT = True` only
if you will **restart the runtime** before 6b.

Colab keeps an event loop in every cell, so when this is on it hands `warmup()` / `measure()`
to a worker thread the same way the run cell does.


In [ ]:
PREFLIGHT = False

if PREFLIGHT:
    from microscope.backends import BackendSpec
    import concurrent.futures
    import gc

    _cfg = configs[-1]
    _opts = dict(_cfg.extra_load_kwargs)
    _opts.update(backend=_cfg.backend, enable_thinking=_cfg.enable_thinking, dtype=_cfg.dtype)
    _s = scenarios[0]
    _before = torch.cuda.memory_reserved() / 1e9

    def _preflight():
        # Worker thread: Colab's kernel loop makes interp-engine refuse warmup() / measure().
        backend = BackendSpec(
            kind="local", model_id=MODEL_ID, options=_opts,
            max_gen_tokens=_cfg.max_gen_tokens,
        ).build()
        try:
            m = backend.measure(_s.prompt("floor"))
            return {
                "engine_backend": backend.handle.backend,
                "can_capture": backend.handle.can_capture,
                "response_mode": backend.response_mode,
                "gen_budget": backend.max_gen_tokens,
                "chosen_letter": m.chosen_letter,
                "parse_ok": m.parse_ok,
                "probability_source": m.probability_source,
                "generated": m.generated,
            }
        finally:
            backend.shutdown()

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        info = pool.submit(_preflight).result()

    print("engine backend :", info["engine_backend"])
    print("can capture    :", info["can_capture"])
    print("response mode  :", info["response_mode"])
    print("gen budget     :", info["gen_budget"])
    print("parsed letter  :", info["chosen_letter"], " expected:", _s.correct_letter)
    print("parse_ok       :", info["parse_ok"], "| source:", info["probability_source"])
    print()
    print("completion tail:", repr(info["generated"][-200:]))
    assert info["parse_ok"], "No answer parsed -- do not start the full run."
    assert info["probability_source"] != "text_truncated", (
        "Truncated mid-reasoning at this budget. Raise max_gen_tokens before running.")
    # Not asserted: one scenario is not a measurement, and a wrong answer here is a result
    # rather than a fault. Only the parse is being checked.
    print("answer correct :", info["chosen_letter"] == _s.correct_letter)
    print("\nPreflight OK.")

    gc.collect()
    torch.cuda.empty_cache()
    _after = torch.cuda.memory_reserved() / 1e9
    print(f"\nGPU reserved: {_before:.1f} GB -> {_after:.1f} GB")
    if _after > _before + 4:
        print("WARNING: memory did not come back. Restart the runtime before the full run.")
else:
    print("Preflight skipped.")


### 6b. Run

Interrupt the kernel or, from the Colab terminal:

```
touch /content/drive/MyDrive/behaviour-microscope/results/thomson-1-thinking/STOP
```

The in-flight measurement finishes, then the run exports what it has. Resume later with the
same `RUN_NAME` (and `START_AT = None`).


In [ ]:
import concurrent.futures
from microscope.experiment import STOP_FILENAME

def _run():
    return run_sweep(configs)

stop_paths = [resolve_run_dir(c) / STOP_FILENAME for c in configs]
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
    future = pool.submit(_run)
    try:
        runs = future.result()
    except KeyboardInterrupt:
        for path in stop_paths:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text("keyboard interrupt\n")
        print("STOP requested. Waiting for the in-flight measurement to flush ...")
        runs = future.result()
runs


### 6c. Export from disk

Rebuild summary, quality gate, and plots from `behavioural.csv` already on Drive. Does **not**
load the model. Run this after a timeout, after STOP, or any time you want numbers from a
partial sweep.


In [ ]:
from microscope.experiment import finalize_run, STOP_FILENAME

runs = {}
for c in configs:
    path = resolve_run_dir(c)
    if not (path / "behavioural.csv").exists():
        print(f"skip {path.name}: no behavioural.csv yet")
        continue
    (path / STOP_FILENAME).unlink(missing_ok=True)
    runs[c.run_name] = finalize_run(path, c)
    print(f"exported {path}")
runs


## 7. Results


In [ ]:
from microscope.experiment import resolve_run_dir

if "runs" not in dir() or not runs:
    runs = {}
    for c in configs:
        path = resolve_run_dir(c)
        if (path / "summary.json").exists():
            runs[c.run_name] = path

for label, path in runs.items():
    s = json.loads((path / "summary.json").read_text())
    q = json.loads((path / "quality_report.json").read_text())
    n = s["behavioural"].get("n_measurements")
    planned = s["behavioural"].get("n_planned")
    print(f"=== {label} — quality gate: {q['overall'].upper()}  ({n}" +
          (f"/{planned}" if planned else "") + " rows) ===")
    for arm, rate in s["behavioural"]["fpar_by_arm"].items():
        acc = s["behavioural"]["accuracy_by_arm"][arm]
        print(f"  {arm:20s} accepts false {rate:5.0%}   accuracy {acc:4.0%}")
    print()


In [ ]:
# Against the other models, if their runs are on this machine.
from pathlib import Path
everything = dict(runs)
roots = [Path("results")]
if "RESULTS_ROOT" in dir():
    roots.append(Path(RESULTS_ROOT))
seen = set()
for root in roots:
    if not root.exists():
        continue
    for p in sorted(root.iterdir()):
        if p in seen or not (p / "summary.json").exists() or p in runs.values():
            continue
        seen.add(p)
        m = json.loads((p / "manifest.json").read_text())
        everything.setdefault(m.get("model", p.name), p)

table = compare_runs(everything)
display(table.style.format("{:.0%}").background_gradient(cmap="Reds", vmin=0, vmax=1))


## 8. Scope

Forced-choice, n=30, England and Wales. Not a measurement of CoCounsel as a deployed system (retrieval, prompting, and product guardrails are out of band). Planned contrast of interest: `partner_confirmed` vs `court`.

If publishing an unfavourable result that names the product, notify Thomson Reuters and offer a right of reply.


## 9. Export


In [ ]:
import shutil
from microscope.experiment import resolve_run_dir

if "runs" not in dir() or not runs:
    runs = {c.run_name: resolve_run_dir(c) for c in configs if (resolve_run_dir(c) / "summary.json").exists()}

for label, path in runs.items():
    archive = shutil.make_archive(str(path), "zip", path)
    print(archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:
        print(f"  already on Drive at {path} ({exc})")
